# PRE-DATA PROCESSING OF DRUG DATASETS

In [9]:
import pandas
import os

In [16]:
# DIRECTORIES
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir) # the notebook is in a subdirectory of the base directory
input_dir = os.path.join(base_dir, 'data', 'input')
print(f"Base directory: {base_dir}")
print(f"Input directory: {input_dir}")

Base directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA
Input directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA\data\input


In [32]:
# FILES
drugdata_file = input_dir + '/vis/vis2024_processed.tsv'
drugdata_df = pandas.read_csv(drugdata_file, sep='\t')
drugdata_df.columns

Index(['CELL_LINE_NAME', 'tissue', 'ANCHOR_ID', 'LIBRARY_ID',
       'ANCHOR_VIABILITY', 'LIBRARY_EMAX', 'SYNERGY_OBS_EMAX',
       'SYNERGY_DELTA_EMAX', 'SYNERGY_DELTA_XMID', 'SYNERGY_DELTA_IC50',
       'SYNERGY_EMAX', 'BLISS_VIS', 'HSA_VIS', 'HSA', 'BLISS', 'SUB_HSA',
       'NO_SYNERGY_VIS', 'NO_BLISS_OR_HSA', 'ANCHOR_NAME', 'LIBRARY_NAME',
       'ANCHOR_GENE_TARGET', 'LIBRARY_GENE_TARGET'],
      dtype='object')

In [29]:
drugdata_df.CELL_LINE_NAME.nunique()

757

In [46]:
# CLEAN CELL LINE NAMES
print(f'Total unique cell lines in dataset before cleaning: {drugdata_df["CELL_LINE_NAME"].nunique()}')
drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.upper()
# Remove '-' or '+' from cell line names
# drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.replace('-', '', regex=False)
drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.replace('+', '', regex=False)
print(f'Total unique cell lines in dataset after cleaning: {drugdata_df["CELL_LINE_NAME"].nunique()}')
# UNIQUE CELL LINES IN DATASET
clines = drugdata_df['CELL_LINE_NAME'].unique().tolist()
clines.sort()

Total unique cell lines in dataset before cleaning: 757
Total unique cell lines in dataset after cleaning: 757


In [47]:
# UNIQUE DRUGS IN DATASET
drug_names = drugdata_df[['ANCHOR_NAME', 'LIBRARY_NAME']].copy()
drug_names['ANCHOR_NAME'] = drug_names['ANCHOR_NAME'].str.upper()
drug_names['LIBRARY_NAME'] = drug_names['LIBRARY_NAME'].str.upper()

# Make list of total drugs in dataset
drug_list = sorted(set(drug_names['ANCHOR_NAME'].tolist() + drug_names['LIBRARY_NAME'].tolist()))
print(f'Total unique drugs in dataset: {len(drug_list)}')

# Save list as dataframe of single column 'drug_name'
drug_df = pandas.DataFrame(drug_list, columns=['drug_name'])
drugnames_file = input_dir + '/vis/drug_names.txt'
drug_df.to_csv(drugnames_file, index=False, header=True)

Total unique drugs in dataset: 29


In [37]:
# UNIQUE TRIPLETS IN DATASET
triplets = drugdata_df[['CELL_LINE_NAME', 'ANCHOR_NAME', 'LIBRARY_NAME']].copy()
triplets = triplets.drop_duplicates()
triplets = triplets.reset_index(drop=True)
print(f'Total unique (cell_line, drugA, drugB) triplets in dataset: {len(triplets)}')

Total unique (cell_line, drugA, drugB) triplets in dataset: 37900


In [40]:
# TISSUE INFORMATION
# Extract tissue information from cell line names
tissue_info = drugdata_df[['tissue', 'CELL_LINE_NAME']].copy()
tissue_info = tissue_info.drop_duplicates()
tissue_info = tissue_info.reset_index(drop=True)
tissue_info.to_csv(input_dir + '/vis/tissue_cline.csv', index=False, header=True)

In [45]:
# SYNERGY DATA
synergies = drugdata_df[['tissue', 'CELL_LINE_NAME', 'ANCHOR_NAME', 'LIBRARY_NAME', 'BLISS']].copy()
rename_dict = {
    'CELL_LINE_NAME': 'cell_line',
    'ANCHOR_NAME': 'drug_name_A',
    'LIBRARY_NAME': 'drug_name_B',
    'BLISS': 'synergy'
}
synergies = synergies.rename(columns=rename_dict)
synergies.to_csv(input_dir + '/vis/synergy_data.csv', index=False, header=True)